# EDA & Limpieza de Datos — BI Sobres Constructora Capital

**Objetivo del proyecto:**  
Procesar y limpiar el reporte de sobres electrónicos exportado desde la plataforma de firmas digitales, con el fin de generar un archivo CSV estructurado que alimenta un dashboard en Power BI.

**Fuente de datos:**  
Archivo `Envelope Recipient Report.csv` — reporte de envíos y firmas por destinatario.

**Salida del proceso:**  
- `Sobres_Limpio.csv` — dataset principal para Power BI  
- `Prediccion_Sobres.csv` — serie histórica + proyección de volumen por tipo de documento

---

## Importación de librerias

Se importan las librerias necesarias para el procesamiento:
- **pandas** y **numpy**: manipulación y análisis de datos tabulares.
- **re**: expresiones regulares para limpiar y extraer texto del campo `Asunto`.
- **unicodedata**: normalización de caracteres especiales (tildes, ñ) para comparaciones de texto.
- **warnings**: suprime alertas no críticas durante la ejecución.

In [123]:
# ============================================================
# EDA & LIMPIEZA COMPLETA - BI Sobres Constructora Capital
# ============================================================

import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import re
import unicodedata
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías cargadas")

✅ Librerías cargadas


## 1. Carga del CSV — Detección del separador

Antes de cargar el archivo se inspecciona la primera línea para detectar automáticamente si el separador es coma (`,`) o punto y coma (`;`).  
Esto es necesario porque el formato de exportación puede variar según la configuración regional del sistema que generó el reporte.

In [124]:
# ============================================================
# 1. CARGA DEL CSV - Detección de separador
# ============================================================

# Primero detectamos el separador
with open(r'C:\Users\diana\OneDrive\Escritorio\Proyecto_1\BI_Sobres\Envelope Recipient Report.csv', 
          encoding='utf-8-sig') as f:
    primera_linea = f.readline()
    print("Primera línea del CSV:")
    print(primera_linea[:200])
    print(f"\nComas encontradas: {primera_linea.count(',')}")
    print(f"Puntos y coma encontrados: {primera_linea.count(';')}")

Primera línea del CSV:
"﻿""Identificación del sobre""";Asunto;Estado;Nombre de remitente;Nombre del destinatario;Orden de envío;Acción;Enviado el (Fecha);Enviado el (Hora);Completado el (Fecha);Completado el (Hora);Tiempo h

Comas encontradas: 0
Puntos y coma encontrados: 16


Una vez confirmado el separador (`;`), se carga el CSV completo en un DataFrame. La codificación `utf-8-sig` elimina el BOM que suelen incluir los archivos exportados desde Excel o herramientas similares. La primera columna se renombra a `ID_Sobre` para identificarla claramente.

In [125]:
# ============================================================
# 1. CARGA DEL CSV
# ============================================================

df = pd.read_csv(
    r'C:\Users\diana\OneDrive\Escritorio\Proyecto_1\BI_Sobres\Envelope Recipient Report.csv',
    sep=';',
    encoding='utf-8-sig',
    on_bad_lines='skip'
)

# Renombrar primera columna
df.rename(columns={df.columns[0]: 'ID_Sobre'}, inplace=True)

print(f"Filas:    {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
print(f"\nColumnas detectadas:")
for i, col in enumerate(df.columns):
    print(f"  {i}: {col}")

Filas:    39,277
Columnas: 17

Columnas detectadas:
  0: ID_Sobre
  1: Asunto
  2: Estado
  3: Nombre de remitente
  4: Nombre del destinatario
  5: Orden de envío
  6: Acción
  7: Enviado el (Fecha)
  8: Enviado el (Hora)
  9: Completado el (Fecha)
  10: Completado el (Hora)
  11: Tiempo hasta finalización (DD:HH:MM)
  12: Fecha de creación (Fecha)
  13: Fecha de creación (Hora)
  14: Nombres de los grupos
  15: Dirección de correo electrónico del destinatario
  16: Dirección de correo electrónico del remitente


## 2. Renombrar columnas

Los nombres originales del reporte vienen en inglés y con formatos poco legibles. Se reemplazan por nombres en español, claros y consistentes, que facilitan el trabajo posterior tanto en Python como en Power BI.  
Cada fila del dataset representa **un destinatario dentro de un sobre**, por lo que un mismo sobre puede aparecer varias veces (una por firmante).

In [126]:
# ============================================================
# 2. RENOMBRAR COLUMNAS
# ============================================================

df.columns = [
    'ID_Sobre', 'Asunto', 'Estado', 'Remitente', 'Destinatario',
    'Orden_Envio', 'Accion', 'Fecha_Envio', 'Hora_Envio',
    'Fecha_Completado', 'Hora_Completado', 'Tiempo_Finalizacion',
    'Fecha_Creacion', 'Hora_Creacion', 'Grupo', 'Email_Destinatario',
    'Email_Remitente'
]

print("✅ Columnas renombradas:")
for i, col in enumerate(df.columns):
    print(f"  {i}: {col}")

✅ Columnas renombradas:
  0: ID_Sobre
  1: Asunto
  2: Estado
  3: Remitente
  4: Destinatario
  5: Orden_Envio
  6: Accion
  7: Fecha_Envio
  8: Hora_Envio
  9: Fecha_Completado
  10: Hora_Completado
  11: Tiempo_Finalizacion
  12: Fecha_Creacion
  13: Hora_Creacion
  14: Grupo
  15: Email_Destinatario
  16: Email_Remitente


## 3. Conversión de fechas

Las columnas de fecha llegan como texto y deben convertirse al tipo `datetime` para poder calcular diferencias de tiempo y filtrar por rangos.  
El parámetro `errors='coerce'` convierte los valores no válidos en `NaT` (Not a Time) en lugar de generar un error, lo que permite continuar el procesamiento.  
Los nulos en `Fecha_Completado` corresponden a sobres que aún están **en proceso** (no firmados por todos los destinatarios).

In [127]:
# ============================================================
# 3. CONVERTIR FECHAS
# ============================================================

df['Fecha_Envio']      = pd.to_datetime(df['Fecha_Envio'],      errors='coerce')
df['Fecha_Completado'] = pd.to_datetime(df['Fecha_Completado'], errors='coerce')
df['Fecha_Creacion']   = pd.to_datetime(df['Fecha_Creacion'],   errors='coerce')

print("✅ Fechas convertidas")
print(f"\nFecha_Envio:      min={df['Fecha_Envio'].min().date()}  max={df['Fecha_Envio'].max().date()}")
print(f"Fecha_Completado: min={df['Fecha_Completado'].min().date()}  max={df['Fecha_Completado'].max().date()}")
print(f"Fecha_Creacion:   min={df['Fecha_Creacion'].min().date()}  max={df['Fecha_Creacion'].max().date()}")
print(f"\nNulos en Fecha_Completado: {df['Fecha_Completado'].isna().sum():,} (sobres sin completar)")

✅ Fechas convertidas

Fecha_Envio:      min=2025-01-12  max=2026-12-04
Fecha_Completado: min=2025-01-12  max=2026-12-04
Fecha_Creacion:   min=2025-10-21  max=2026-05-02

Nulos en Fecha_Completado: 28,335 (sobres sin completar)


## 4. Verificación de estados y acciones

Se analiza la distribución de los dos campos categóricos principales:
- **Estado**: aplica al nivel del sobre completo (`Completado`, `Enviado`, `Anulado`).
- **Acción**: aplica al nivel del destinatario individual (`Firmado`, `Creado`, `Enviado`, etc.).

También se cruzan los nulos de `Fecha_Completado` con estos campos para entender si los valores faltantes tienen un patrón lógico (ej.: sobres enviados que aún no han sido firmados).

In [128]:
# ============================================================
# 4. VERIFICACIÓN DE ESTADOS Y ACCIONES
# ============================================================

print("=" * 50)
print("DISTRIBUCIÓN DE ESTADOS (nivel sobre)")
print("=" * 50)
print(df.groupby('Estado')['ID_Sobre'].nunique().sort_values(ascending=False))

print("\n" + "=" * 50)
print("DISTRIBUCIÓN DE ACCIONES (nivel destinatario)")
print("=" * 50)
print(df['Accion'].value_counts())

print("\n" + "=" * 50)
print("NULOS EN FECHA_COMPLETADO POR ESTADO")
print("=" * 50)
print(df[df['Fecha_Completado'].isna()].groupby('Estado')['ID_Sobre'].nunique().sort_values(ascending=False))

print("\n" + "=" * 50)
print("NULOS EN FECHA_COMPLETADO POR ACCIÓN")
print("=" * 50)
print(df[df['Fecha_Completado'].isna()]['Accion'].value_counts())

DISTRIBUCIÓN DE ESTADOS (nivel sobre)
Estado
Completado    5969
Enviado       1541
Anulado        865
Correcto        39
Declinado       33
Name: ID_Sobre, dtype: int64

DISTRIBUCIÓN DE ACCIONES (nivel destinatario)
Accion
Completado       29980
Creado            6628
Enviado           2103
Entregado          493
AutoResponded       40
Declinado           33
Name: count, dtype: int64

NULOS EN FECHA_COMPLETADO POR ESTADO
Estado
Completado    5000
Enviado       1541
Anulado        865
Correcto        39
Declinado       33
Name: ID_Sobre, dtype: int64

NULOS EN FECHA_COMPLETADO POR ACCIÓN
Accion
Completado       19038
Creado            6628
Enviado           2103
Entregado          493
AutoResponded       40
Declinado           33
Name: count, dtype: int64


## 5. Fechas de completado y tiempo en días

Dado que un sobre tiene múltiples firmantes, se necesitan **dos niveles de fecha**:
- `Fecha_Completado_Firmante`: fecha en que cada destinatario firmó (nivel fila).
- `Fecha_Completado_Sobre`: fecha en que el **último** firmante completó el sobre, calculada como el máximo por `ID_Sobre`.

Con estas fechas se calcula `Tiempo_Dias`, que mide cuántos días tardó un sobre desde que fue enviado hasta que fue completado por todos los firmantes. Este campo solo se calcula para sobres con estado `Completado`.

In [129]:
# ============================================================
# 5. FECHAS DE COMPLETADO Y TIEMPO EN DÍAS
# ============================================================

# Fecha completado del sobre = fecha máxima entre todos los firmantes
fecha_sobre = (df[df['Fecha_Completado'].notna()]
               .groupby('ID_Sobre')['Fecha_Completado']
               .max()
               .reset_index()
               .rename(columns={'Fecha_Completado': 'Fecha_Completado_Sobre'}))

df = df.merge(fecha_sobre, on='ID_Sobre', how='left')

# Renombrar la individual
df.rename(columns={'Fecha_Completado': 'Fecha_Completado_Firmante'}, inplace=True)

# Tiempo en días = solo para sobres Completados
# Desde Fecha_Envio hasta Fecha_Completado_Sobre
df['Tiempo_Dias'] = None
mask = df['Estado'] == 'Completado'
df.loc[mask, 'Tiempo_Dias'] = (
    df.loc[mask, 'Fecha_Completado_Sobre'] - df.loc[mask, 'Fecha_Envio']
).dt.days.round(2)

print("✅ Fechas y tiempos calculados")
print(f"\nFecha_Completado_Sobre nulos:    {df['Fecha_Completado_Sobre'].isna().sum():,}")
print(f"Fecha_Completado_Firmante nulos: {df['Fecha_Completado_Firmante'].isna().sum():,}")
print(f"\n--- Estadísticas Tiempo_Dias (solo Completados) ---")
completados = df[df['Estado'] == 'Completado']['Tiempo_Dias'].dropna()
print(f"Promedio: {completados.mean():.2f} días")
print(f"Mediana:  {completados.median():.2f} días")
print(f"Máximo:   {completados.max():.2f} días")
print(f"Mínimo:   {completados.min():.2f} días")

✅ Fechas y tiempos calculados

Fecha_Completado_Sobre nulos:    20,276
Fecha_Completado_Firmante nulos: 28,335

--- Estadísticas Tiempo_Dias (solo Completados) ---
Promedio: 72.96 días
Mediana:  31.00 días
Máximo:   690.00 días
Mínimo:   -333.00 días


## Investigación: tiempos anómalos

Se detectaron valores de `Tiempo_Dias` fuera de rango que indican un problema en la lectura de fechas:
- **Tiempos negativos**: la fecha de completado es anterior a la de envío (imposible en la realidad).
- **Tiempos mayores a 365 días**: implican que el sobre tardó más de un año en completarse, lo que es sospechoso.

Ambos síntomas apuntan a que las fechas no se parsearon correctamente debido al formato del CSV.

In [130]:
# ============================================================
# INVESTIGACIÓN DE TIEMPOS ANÓMALOS
# ============================================================

print("=" * 50)
print("TIEMPOS NEGATIVOS")
print("=" * 50)
negativos = df[df['Tiempo_Dias'] < 0][['ID_Sobre','Estado','Fecha_Envio','Fecha_Completado_Sobre','Tiempo_Dias']].drop_duplicates('ID_Sobre')
print(f"Sobres con tiempo negativo: {len(negativos):,}")
print(negativos.head(5).to_string())

print("\n" + "=" * 50)
print("TIEMPOS MAYORES A 365 DÍAS")
print("=" * 50)
largos = df[df['Tiempo_Dias'] > 365][['ID_Sobre','Estado','Fecha_Envio','Fecha_Completado_Sobre','Tiempo_Dias']].drop_duplicates('ID_Sobre')
print(f"Sobres con más de 365 días: {len(largos):,}")
print(largos.head(5).to_string())

TIEMPOS NEGATIVOS
Sobres con tiempo negativo: 93
                                   ID_Sobre      Estado Fecha_Envio Fecha_Completado_Sobre Tiempo_Dias
5856   e609c0d9-1fb1-4b1d-a61a-31c6f1198e0b  Completado  2026-12-04             2026-02-05      -302.0
6606   086e839a-87c9-4c2b-848e-54388829af36  Completado  2026-10-04             2026-03-05      -213.0
7290   bcd63f55-fc0f-44e6-a262-193dc868cf52  Completado  2026-08-04             2026-03-05      -152.0
7321   af4bdfff-eb2e-4ab5-95ed-21f6075d148d  Completado  2026-08-04             2026-03-05      -152.0
14945  dee00a60-d8e3-409b-b7b1-9c00dbb253b9  Completado  2026-12-03             2026-01-04      -333.0

TIEMPOS MAYORES A 365 DÍAS
Sobres con más de 365 días: 20
                                   ID_Sobre      Estado Fecha_Envio Fecha_Completado_Sobre Tiempo_Dias
31124  0ddfe052-5bfe-4baf-a475-55b82f60864e  Completado  2025-11-12             2026-12-02       385.0
31184  7cf02e51-2f3f-477a-be47-9820a6313cc8  Completado  2025-11-12 

## Corrección: formato de fecha DD/MM/YYYY

El CSV usa el formato colombiano `DD/MM/YYYY`, pero `pd.to_datetime()` por defecto asume `MM/DD/YYYY`. Esto causa que días como `05/03` se interpreten como marzo 5 en lugar de 3 de mayo, generando los tiempos negativos y exagerados.  
Se fuerza el formato explícitamente con `format='%d/%m/%Y'` y se recalculan las fechas del sobre y el `Tiempo_Dias`.

In [131]:
# ============================================================
# CORRECCIÓN - Fechas en formato DD/MM/YYYY
# ============================================================

df['Fecha_Envio']      = pd.to_datetime(df['Fecha_Envio'],      format='%d/%m/%Y', errors='coerce')
df['Fecha_Completado_Firmante'] = pd.to_datetime(df['Fecha_Completado_Firmante'], format='%d/%m/%Y', errors='coerce')
df['Fecha_Creacion']   = pd.to_datetime(df['Fecha_Creacion'],   format='%d/%m/%Y', errors='coerce')

# Recalcular Fecha_Completado_Sobre
fecha_sobre = (df[df['Fecha_Completado_Firmante'].notna()]
               .groupby('ID_Sobre')['Fecha_Completado_Firmante']
               .max()
               .reset_index()
               .rename(columns={'Fecha_Completado_Firmante': 'Fecha_Completado_Sobre'}))

df = df.drop(columns=['Fecha_Completado_Sobre'])
df = df.merge(fecha_sobre, on='ID_Sobre', how='left')

# Recalcular Tiempo_Dias
df['Tiempo_Dias'] = None
mask = df['Estado'] == 'Completado'
df.loc[mask, 'Tiempo_Dias'] = (
    df.loc[mask, 'Fecha_Completado_Sobre'] - df.loc[mask, 'Fecha_Envio']
).dt.days

print("✅ Fechas corregidas con formato DD/MM/YYYY")
print(f"\nFecha_Envio:      min={df['Fecha_Envio'].min().date()}  max={df['Fecha_Envio'].max().date()}")
print(f"Fecha_Completado: min={df['Fecha_Completado_Sobre'].min().date()}  max={df['Fecha_Completado_Sobre'].max().date()}")

print(f"\n--- Estadísticas Tiempo_Dias (solo Completados) ---")
completados = df[df['Estado'] == 'Completado']['Tiempo_Dias'].dropna()
print(f"Promedio: {completados.mean():.2f} días")
print(f"Mediana:  {completados.median():.2f} días")
print(f"Máximo:   {completados.max():.2f} días")
print(f"Mínimo:   {completados.min():.2f} días")

print(f"\nTiempos negativos:      {(completados < 0).sum():,}")
print(f"Tiempos > 365 días:     {(completados > 365).sum():,}")

✅ Fechas corregidas con formato DD/MM/YYYY

Fecha_Envio:      min=2025-01-12  max=2026-12-04
Fecha_Completado: min=2025-01-12  max=2026-12-04

--- Estadísticas Tiempo_Dias (solo Completados) ---
Promedio: 72.96 días
Mediana:  31.00 días
Máximo:   690.00 días
Mínimo:   -333.00 días

Tiempos negativos:      428
Tiempos > 365 días:     97


## Investigación: fechas crudas en el CSV original

Para diagnosticar el problema de raíz, se vuelve a leer el CSV sin conversión de fechas y se inspeccionan los valores crudos de los sobres problemáticos. Esto permite confirmar si el problema es el parseo de Python o si el archivo fuente tiene datos inconsistentes.

In [132]:
# ============================================================
# INVESTIGACIÓN - Fechas crudas en CSV original
# ============================================================

# Releer solo las columnas de fechas sin convertir
df_raw = pd.read_csv(
    r'C:\Users\diana\Documents\BI_Sobres\Envelope Recipient Report.csv',
    sep=';',
    encoding='utf-8-sig',
    on_bad_lines='skip'
)
df_raw.rename(columns={df_raw.columns[0]: 'ID_Sobre'}, inplace=True)
df_raw.columns = [
    'ID_Sobre', 'Asunto', 'Estado', 'Remitente', 'Destinatario',
    'Orden_Envio', 'Accion', 'Fecha_Envio', 'Hora_Envio',
    'Fecha_Completado', 'Hora_Completado', 'Tiempo_Finalizacion',
    'Fecha_Creacion', 'Hora_Creacion', 'Grupo', 'Email_Destinatario',
    'Email_Remitente'
]

ids_negativos = [
    'e609c0d9-1fb1-4b1d-a61a-31c6f1198e0b',
    'dee00a60-d8e3-409b-b7b1-9c00dbb253b9'
]

print("--- Fechas CRUDAS de sobres con tiempo negativo ---")
muestra = df_raw[df_raw['ID_Sobre'].isin(ids_negativos)][
    ['ID_Sobre','Fecha_Envio','Fecha_Completado']
].drop_duplicates('ID_Sobre')
print(muestra.to_string())

print("\n--- Muestra general de fechas crudas ---")
print(df_raw[['Fecha_Envio','Fecha_Completado']].dropna().head(10).to_string())

--- Fechas CRUDAS de sobres con tiempo negativo ---
                                   ID_Sobre Fecha_Envio Fecha_Completado
5856   e609c0d9-1fb1-4b1d-a61a-31c6f1198e0b  12/04/2026        2/05/2026
14945  dee00a60-d8e3-409b-b7b1-9c00dbb253b9  12/03/2026        1/04/2026

--- Muestra general de fechas crudas ---
    Fecha_Envio Fecha_Completado
13    2/05/2026        2/05/2026
15    2/05/2026        2/05/2026
18    2/05/2026        2/05/2026
37    2/05/2026        2/05/2026
59    2/05/2026        2/05/2026
81    2/05/2026        2/05/2026
110   2/05/2026        2/05/2026
115   2/05/2026        2/05/2026
116   2/05/2026        2/05/2026
136   2/05/2026        2/05/2026


## Corrección definitiva: `dayfirst=True`

La solución definitiva es usar el parámetro `dayfirst=True` en `pd.to_datetime()`, que le indica a pandas que el primer componente de la fecha es el día. Esta es la forma más robusta de manejar fechas en formato colombiano/europeo, ya que funciona incluso cuando el formato no es completamente consistente en el CSV.

In [133]:
# ============================================================
# CORRECCIÓN DEFINITIVA - Fechas con dayfirst=True
# ============================================================

df['Fecha_Envio']               = pd.to_datetime(df_raw['Fecha_Envio'],      dayfirst=True, errors='coerce')
df['Fecha_Completado_Firmante'] = pd.to_datetime(df_raw['Fecha_Completado'], dayfirst=True, errors='coerce')
df['Fecha_Creacion']            = pd.to_datetime(df_raw['Fecha_Creacion'],   dayfirst=True, errors='coerce')

# Recalcular Fecha_Completado_Sobre
df = df.drop(columns=['Fecha_Completado_Sobre'])
fecha_sobre = (df[df['Fecha_Completado_Firmante'].notna()]
               .groupby('ID_Sobre')['Fecha_Completado_Firmante']
               .max()
               .reset_index()
               .rename(columns={'Fecha_Completado_Firmante': 'Fecha_Completado_Sobre'}))
df = df.merge(fecha_sobre, on='ID_Sobre', how='left')

# Recalcular Tiempo_Dias
df['Tiempo_Dias'] = None
mask = df['Estado'] == 'Completado'
df.loc[mask, 'Tiempo_Dias'] = (
    df.loc[mask, 'Fecha_Completado_Sobre'] - df.loc[mask, 'Fecha_Envio']
).dt.days

print("✅ Fechas corregidas con dayfirst=True")
completados = df[df['Estado'] == 'Completado']['Tiempo_Dias'].dropna()
print(f"Promedio: {completados.mean():.2f} días")
print(f"Mediana:  {completados.median():.2f} días")
print(f"Máximo:   {completados.max():.2f} días")
print(f"Mínimo:   {completados.min():.2f} días")
print(f"\nTiempos negativos:  {(completados < 0).sum():,}")
print(f"Tiempos > 365 días: {(completados > 365).sum():,}")

✅ Fechas corregidas con dayfirst=True
Promedio: 16.08 días
Mediana:  10.00 días
Máximo:   165.00 días
Mínimo:   0.00 días

Tiempos negativos:  0
Tiempos > 365 días: 0


## 6. Clasificación del tipo de documento

El campo `Asunto` contiene un código de 6 dígitos al inicio (ej.: `120101`) que identifica el tipo de documento del sobre. Se construye una función de clasificación que:
1. Extrae el código de los primeros 6 caracteres.
2. Detecta **devoluciones** cuando el código tiene el formato `XXXXXX.N` (donde N es el número de devolución).
3. Identifica **errores de marcación** (asteriscos, espacios faltantes, prefijos incorrectos).
4. Mapea cada código a su nombre descriptivo (ej.: `120101` → `Cierre de venta`).

Las columnas resultantes son: `Tipo_Documento`, `Codigo`, `Marcacion_Error`, `Num_Devoluciones` y `Tipo_Error`.

In [134]:
# ============================================================
# 6. TIPOS DE DOCUMENTO
# ============================================================

tipos_doc = {
    '120101': 'Cierre de venta',
    '120102': 'Promesa de compraventa',
    '120103': 'Otrosí',
    '120104': 'Cesión de derechos',
    '120106': 'Otrosí ajuste de salarios',
    '120107': 'Contrato mera tenencia',
    '120112': 'Contratos de vinculación',
    '120115': 'Exoneración impuesto rentas y registro',
    '120124': 'Tipo pendiente por clasificar',
    '120125': 'Tipo pendiente por clasificar',
    '120108': 'Tipo pendiente por clasificar',
    '120109': 'Tipo pendiente por clasificar',
    '120110': 'Tipo pendiente por clasificar',
    '120105': 'Tipo pendiente por clasificar',
}

def clasificar_documento(asunto):
    asunto = str(asunto).strip()
    codigo6 = asunto[:6]
    if codigo6 in ['12010.', 'FORMUL', 'Comple']:
        return 'Sobre mal marcado', codigo6, False, 0, None
    match = re.match(r'^(\d{6})\.(\d)(\d?)\s', asunto)
    if match:
        cod      = match.group(1)
        num_dev  = int(match.group(2))
        tipo_err = int(match.group(3)) if match.group(3) else None
        tipo     = tipos_doc.get(cod, 'Tipo pendiente por clasificar')
        return tipo, cod, True, num_dev, tipo_err
    if re.match(r'^\d{6}\*', asunto):
        cod = asunto[:6]
        return tipos_doc.get(cod, 'Tipo pendiente por clasificar'), cod, True, 0, None
    if re.match(r'^\d{6}[A-Za-záéíóúÁÉÍÓÚñÑ]', asunto):
        cod = asunto[:6]
        return tipos_doc.get(cod, 'Tipo pendiente por clasificar'), cod, True, 0, None
    cod  = asunto[:6]
    tipo = tipos_doc.get(cod, 'Tipo pendiente por clasificar')
    return tipo, cod, False, 0, None

resultado = df['Asunto'].apply(clasificar_documento)
df['Tipo_Documento']   = resultado.apply(lambda x: x[0])
df['Codigo']           = resultado.apply(lambda x: x[1])
df['Marcacion_Error']  = resultado.apply(lambda x: x[2])
df['Num_Devoluciones'] = resultado.apply(lambda x: x[3])
df['Tipo_Error']       = resultado.apply(lambda x: x[4])

print("✅ Tipos de documento clasificados")
print(f"\n{df['Tipo_Documento'].value_counts().to_string()}")

✅ Tipos de documento clasificados

Tipo_Documento
Cierre de venta                           10878
Otrosí                                    10245
Otrosí ajuste de salarios                  4504
Tipo pendiente por clasificar              3639
Promesa de compraventa                     3319
Contratos de vinculación                   3047
Cesión de derechos                         2615
Contrato mera tenencia                      755
Exoneración impuesto rentas y registro      266
Sobre mal marcado                             9


## Análisis: promedio de días por tipo de documento

Con los tipos de documento clasificados y los tiempos calculados, se obtiene un resumen estadístico (promedio, mediana, máximo, mínimo y total de sobres) para cada código. Este análisis permite identificar qué tipos de contrato tardan más en completarse y establecer límites de alerta para el semáforo.

In [135]:
# ============================================================
# ANÁLISIS - Promedio días por código de documento
# ============================================================

# Tomar un registro por sobre (evitar duplicados por destinatario)
df_sobres = df[df['Estado'] == 'Completado'].drop_duplicates('ID_Sobre')

promedio_codigo = (df_sobres.groupby(['Codigo', 'Tipo_Documento'])['Tiempo_Dias']
                   .agg(['mean', 'median', 'max', 'min', 'count'])
                   .round(2)
                   .reset_index()
                   .rename(columns={
                       'mean':   'Promedio_Dias',
                       'median': 'Mediana_Dias',
                       'max':    'Max_Dias',
                       'min':    'Min_Dias',
                       'count':  'Total_Sobres'
                   })
                   .sort_values('Promedio_Dias', ascending=False))

print("=" * 60)
print("PROMEDIO DE DÍAS POR TIPO DE DOCUMENTO")
print("=" * 60)
print(promedio_codigo.to_string(index=False))

PROMEDIO DE DÍAS POR TIPO DE DOCUMENTO
Codigo                         Tipo_Documento Promedio_Dias Mediana_Dias Max_Dias Min_Dias  Total_Sobres
120107                 Contrato mera tenencia     26.044118         21.0      112        1            68
120102                 Promesa de compraventa     25.228125         16.0      109        0           320
120103                                 Otrosí     23.575984         16.5      165        0          1474
120109          Tipo pendiente por clasificar     21.692308         11.0       67        0            13
120104                     Cesión de derechos     21.611765         14.0      119        0           255
120106              Otrosí ajuste de salarios     18.749077         16.0      110        1           542
120124          Tipo pendiente por clasificar     11.121739          9.0       73        0           345
120112               Contratos de vinculación     10.664815          6.0      113        0           540
120101          

## 7. Extracción de proyecto, torre y unidad

El campo `Asunto` contiene texto libre que incluye el nombre del proyecto, la torre y la unidad (apartamento o parqueadero). Se construyen dos funciones de extracción con expresiones regulares:
- `extraer_proyecto()`: elimina el código del inicio y extrae el nombre del proyecto antes de la indicación de torre (`T1`, `T2`, etc.).
- `extraer_torre_unidad()`: busca patrones como `T1`, `APT 401`, `PARQ 10` para identificar torre y unidad.

Estos campos son fundamentales para poder filtrar y segmentar los sobres por proyecto en el dashboard de Power BI.

In [136]:
# ============================================================
# 7. EXTRAER PROYECTO, TORRE Y UNIDAD
# ============================================================

def normalizar(texto):
    if pd.isna(texto):
        return ''
    texto = str(texto).lower().strip()
    return ''.join(c for c in unicodedata.normalize('NFD', texto)
                   if unicodedata.category(c) != 'Mn')

def extraer_proyecto(asunto):
    asunto = str(asunto).strip()
    texto  = re.sub(r'^\d{6}[\.\d\*]*\s*', '', asunto).strip()
    if re.match(r'(?i)exoner', texto):
        match = re.search(r'(?i)rentas\s+(?:y\s+registro\s+)?([A-Za-záéíóúÁÉÍÓÚñÑ\s]+?)\s+T\d+', texto)
        if match:
            return match.group(1).strip()
    if re.match(r'(?i)reintegro', texto):
        return 'PENDIENTE - REINTEGRO'
    texto = re.sub(r'\s+atado.*', '', texto, flags=re.IGNORECASE)
    texto = re.sub(r'([A-Za-záéíóúÁÉÍÓÚñÑ])T(\d+)', r'\1 T\2', texto)
    match = re.match(r'^(.+?)\s+T\d+[\s\-]', texto)
    if match:
        return match.group(1).strip()
    match2 = re.match(r'^(.+?)\s*(PARQ|APT|APTO|PQ)', texto, re.IGNORECASE)
    if match2:
        return match2.group(1).strip()
    return texto.strip()

def extraer_torre_unidad(asunto):
    asunto = str(asunto)
    asunto = re.sub(r'([A-Za-záéíóúÁÉÍÓÚñÑ])T(\d+)', r'\1 T\2', asunto)
    torre  = None
    unidad = None
    t = re.search(r'\bT(\d+)\b', asunto)
    if t:
        torre = f"T{t.group(1)}"
    u = re.search(r'(APT|PARQ|PQ-MOTO|APTO)\s*([\w\-]+)', asunto)
    if u:
        unidad = f"{u.group(1)} {u.group(2)}"
    return torre, unidad

df['Proyecto']      = df['Asunto'].apply(extraer_proyecto)
torres_unidades     = df['Asunto'].apply(extraer_torre_unidad)
df['Torre']         = torres_unidades.apply(lambda x: x[0])
df['Unidad']        = torres_unidades.apply(lambda x: x[1])

print("✅ Proyecto, Torre y Unidad extraídos")
print(f"\nProyectos únicos: {df['Proyecto'].nunique():,}")
print(f"\nTop 10 proyectos por registros:")
print(df['Proyecto'].value_counts().head(10).to_string())
print(f"\nTorres únicas: {df['Torre'].nunique()}")
print(df['Torre'].value_counts().head(10).to_string())

✅ Proyecto, Torre y Unidad extraídos

Proyectos únicos: 127

Top 10 proyectos por registros:
Proyecto
San Agustin        4164
Puerto Vallarta    3802
Villa Vento        3055
Tayrona            3006
Velvet             2210
Luna del Cerro     2068
Puerto Capri       1614
Verde Olivo        1594
Nature Aqua        1540
Entrebosques       1442

Torres únicas: 12
Torre
T2     12533
T1     10002
T3      8353
T4      3628
T5      2572
T7       435
T6       201
T11      128
T9       105
T10       95


## 8. Mapeo oficial de proyectos

La extracción del nombre del proyecto desde texto libre genera variaciones del mismo nombre (ej.: `"Puerto Capri torre 4"`, `"Puerto Capri T5"`, `"puerto capri  t5"`). Se construye un diccionario `mapeo_proyectos` que normaliza todas las variaciones a los nombres oficiales de los proyectos de Constructora Capital.  
La normalización se hace con la función `normalizar()` que convierte el texto a minúsculas y elimina tildes, garantizando comparaciones robustas.

In [137]:
# ============================================================
# 8. MAPEO OFICIAL DE PROYECTOS
# ============================================================

proyectos_oficiales = [
    'Puerto Ventura', 'Puerto Vallarta', 'Puerto Sereno', 'Puerto Capri',
    'San Agustin', 'Cocuy', 'Tayrona', 'Entrebosques', 'Monte Rosa',
    'Valle Rosa', 'Bosque Rosa', 'Urbanity', 'Villa Vento', 'Nature Bio',
    'Nature Aqua', 'Bosque del Rio', 'Arista', 'Acento', 'Murano',
    'Bulevar Verde', 'Velvet', 'Indigo', 'Lino', 'Verde Olivo',
    'Verde Silvestre', 'Luna del Cerro', 'Artesia', 'Artiko',
    'Villa Fuerte', 'Turquesa', 'Biocity', 'Biocity Grand',
    'Biocity Epic', 'Metropolitan', 'Entrecerros'
]

mapeo_proyectos = {
    'san agustin': 'San Agustin', 'san agustín': 'San Agustin',
    'puerto vallarta': 'Puerto Vallarta', 'puerto sereno': 'Puerto Sereno',
    'puerto capri': 'Puerto Capri', 'puerto capri torre 4': 'Puerto Capri',
    'puerto capri t5': 'Puerto Capri', 'puerto ventura': 'Puerto Ventura',
    'cocuy': 'Cocuy', 'coucy': 'Cocuy', 'tayrona': 'Tayrona',
    'entrebosques': 'Entrebosques', 'bosque del rio': 'Bosque del Rio',
    'bosque del río': 'Bosque del Rio', 'bosque rosa': 'Bosque Rosa',
    'valle rosa': 'Valle Rosa', 'monte rosa': 'Monte Rosa',
    'urbanity': 'Urbanity', 'rs urbanity': 'Urbanity',
    'villa vento': 'Villa Vento', 'nature bio': 'Nature Bio',
    'nature aqua': 'Nature Aqua', 'arista': 'Arista', 'acento': 'Acento',
    'murano': 'Murano', 'bulevar verde': 'Bulevar Verde', 'velvet': 'Velvet',
    'indigo': 'Indigo', 'lino': 'Lino', 'verde olivo': 'Verde Olivo',
    'verde silvestre': 'Verde Silvestre', 'luna del cerro': 'Luna del Cerro',
    'artesia': 'Artesia', 'artiko': 'Artiko', 'villa fuerte': 'Villa Fuerte',
    'turquesa': 'Turquesa', 'biocity': 'Biocity', 'biocity grand': 'Biocity Grand',
    'metropolitan': 'Metropolitan', 'entrecerros': 'Entrecerros',
    'pendiente - reintegro': 'PENDIENTE - REINTEGRO',
}

def mapear_proyecto(proyecto):
    key = normalizar(str(proyecto).strip())
    return mapeo_proyectos.get(key, 'SIN PROYECTO')

df['Proyecto'] = df['Proyecto'].apply(mapear_proyecto)

print("✅ Proyectos mapeados a nombres oficiales")
print(f"\nProyectos únicos: {df['Proyecto'].nunique()}")
print(f"\nDistribución:")
print(df['Proyecto'].value_counts().to_string())

✅ Proyectos mapeados a nombres oficiales

Proyectos únicos: 36

Distribución:
Proyecto
San Agustin              4804
Puerto Vallarta          3985
Tayrona                  3184
Villa Vento              3106
Velvet                   2210
Luna del Cerro           2150
Puerto Capri             1695
Verde Olivo              1599
Nature Aqua              1540
Cocuy                    1516
Entrebosques             1513
Turquesa                 1252
Nature Bio               1202
Bulevar Verde            1186
Indigo                   1033
SIN PROYECTO              949
Bosque del Rio            881
Verde Silvestre           873
Puerto Sereno             826
Lino                      719
Arista                    624
Villa Fuerte              457
Monte Rosa                371
Artesia                   300
Urbanity                  268
Artiko                    188
Murano                    169
Bosque Rosa               153
Valle Rosa                149
Acento                    133
PENDIENTE - R

### Investigación y corrección de registros SIN PROYECTO

Después del mapeo inicial quedan algunos registros sin proyecto asignado. Se inspeccionan manualmente los asuntos únicos para identificar patrones no cubiertos. Se realizan tres rondas de corrección iterativa:
1. **Primera corrección**: casos con variaciones tipográficas adicionales.
2. **Segunda corrección**: función con fallback al campo `Remitente` cuando el asunto no es suficiente.
3. **Corrección definitiva**: los últimos registros se corrigen directamente por condición en el campo `Asunto`.

In [138]:
# ============================================================
# INVESTIGACIÓN - Registros SIN PROYECTO
# ============================================================

sin_proyecto = df[df['Proyecto'] == 'SIN PROYECTO'][['Asunto', 'Remitente', 'Estado']].drop_duplicates('Asunto')
print(f"Asuntos únicos sin proyecto: {len(sin_proyecto)}")
print(sin_proyecto.to_string())

Asuntos únicos sin proyecto: 173
                                                                                                Asunto                       Remitente      Estado
8040                                                              120112 San Agustín Torre 1 Apto 0302      Sala de Ventas San Agustín  Completado
8055                                                              120112 San Agustín Torre 1 Apto 0301      Sala de Ventas San Agustín  Completado
8164                                                               120112 San Agustín Torre 1 Apt 0228      Sala de Ventas San Agustín  Completado
8179                                                              120112 San Agustín Torre 1 Apto 0205      Sala de Ventas San Agustín  Completado
8209                                                                 120101 Luna del Cerro T 1 APT 222   Sala de Ventas Luna del Cerro  Completado
8234                                                               120112 San Agustín

In [139]:
# ============================================================
# COMPLETAR MAPEO - Casos SIN PROYECTO
# ============================================================

mapeo_adicional = {
    'san agustín torre 1'            : 'San Agustin',
    'reformas torre 5'               : 'Bulevar Verde',
    'reforma torre 5'                : 'Bulevar Verde',
    'cocuy t'                        : 'Cocuy',
    'alianza - bosque del río'       : 'Bosque del Rio',
    '10 bosque del río'              : 'Bosque del Rio',
    'puerto capri  t5'               : 'Puerto Capri',
}

mapeo_proyectos.update(mapeo_adicional)
df['Proyecto'] = df['Asunto'].apply(extraer_proyecto).apply(mapear_proyecto)

print("✅ Mapeo completado")
print(f"SIN PROYECTO restantes: {(df['Proyecto'] == 'SIN PROYECTO').sum():,}")

✅ Mapeo completado
SIN PROYECTO restantes: 406


In [140]:
# ============================================================
# MAPEO FINAL - Últimos casos SIN PROYECTO
# ============================================================

mapeo_final = {
    'formulario de vinculacion apt 1201 lino': 'Lino',
    'luna del cerro': 'Luna del Cerro',
}

mapeo_proyectos.update(mapeo_final)

def mapear_proyecto_con_fallback(row):
    proyecto_asunto = extraer_proyecto(str(row['Asunto']))
    key = normalizar(proyecto_asunto.strip())
    if key in mapeo_proyectos:
        return mapeo_proyectos[key]
    remitente = str(row['Remitente']).strip()
    remitente = re.sub(r'(?i)sala de ventas\s*', '', remitente)
    remitente = re.sub(r'(?i)sala de negocios\s*', '', remitente)
    remitente = re.sub(r'(?i)tramitador-\s*', '', remitente).strip()
    key_rem = normalizar(remitente)
    if key_rem in mapeo_proyectos:
        return mapeo_proyectos[key_rem]
    return 'SIN PROYECTO'

df['Proyecto'] = df.apply(mapear_proyecto_con_fallback, axis=1)

print("✅ Mapeo final completado")
print(f"SIN PROYECTO restantes: {(df['Proyecto'] == 'SIN PROYECTO').sum():,}")

✅ Mapeo final completado
SIN PROYECTO restantes: 11


In [141]:
# ============================================================
# MAPEO DEFINITIVO - Últimos 7 registros
# ============================================================

mask_verde_olivo = df['Asunto'].str.strip() == '120107 T1 APT 1203'
mask_luna = df['Asunto'].str.startswith('Completado 120115')

df.loc[mask_verde_olivo, 'Proyecto'] = 'Verde Olivo'
df.loc[mask_luna, 'Proyecto']        = 'Luna del Cerro'

print("✅ Mapeo definitivo completado")
print(f"SIN PROYECTO restantes: {(df['Proyecto'] == 'SIN PROYECTO').sum():,}")
print(f"\nProyectos únicos: {df['Proyecto'].nunique()}")
print(df['Proyecto'].value_counts().to_string())

✅ Mapeo definitivo completado
SIN PROYECTO restantes: 9

Proyectos únicos: 36
Proyecto
San Agustin              4847
Puerto Vallarta          3995
Tayrona                  3188
Villa Vento              3110
Velvet                   2215
Luna del Cerro           2179
Bulevar Verde            1751
Puerto Capri             1715
Verde Olivo              1611
Nature Aqua              1559
Cocuy                    1520
Entrebosques             1513
Turquesa                 1252
Nature Bio               1210
Indigo                   1033
Bosque del Rio            898
Verde Silvestre           878
Puerto Sereno             826
Lino                      722
Arista                    624
Villa Fuerte              457
Monte Rosa                371
Urbanity                  349
Artesia                   300
Artiko                    188
Murano                    169
Bosque Rosa               158
Valle Rosa                154
Acento                    133
Metropolitan              104
Puerto Ventur

## 9. Flags de gestión: seguimiento y calidad

Se crean dos columnas booleanas para categorizar sobres que requieren atención:
- **`Requiere_Seguimiento`**: sobres que **no están** en estado `Completado` ni `Anulado`, es decir, están activos y pendientes de firma.
- **`Requiere_Calidad`**: sobres que tienen un error de marcación en el campo `Asunto` (identificados en el paso 6).

Estos flags son los que alimentan las vistas de gestión operativa en el dashboard de Power BI.

In [142]:
# ============================================================
# 9. REQUIERE GESTIÓN
# ============================================================

# Gestión de seguimiento: sobres no completados ni anulados
df['Requiere_Seguimiento'] = ~df['Estado'].isin(['Completado', 'Anulado'])

# Gestión de calidad: sobres con error de marcación
df['Requiere_Calidad'] = df['Marcacion_Error'] == True

print("✅ Flags de gestión calculados")
print(f"\nRequiere Seguimiento: {df['Requiere_Seguimiento'].sum():,}")
print(f"Requiere Calidad:     {df['Requiere_Calidad'].sum():,}")
print(f"\nDistribución Requiere_Seguimiento por Estado:")
print(df[df['Requiere_Seguimiento']]['Estado'].value_counts())

✅ Flags de gestión calculados

Requiere Seguimiento: 7,513
Requiere Calidad:     3,224

Distribución Requiere_Seguimiento por Estado:
Estado
Enviado      7165
Correcto      193
Declinado     155
Name: count, dtype: int64


## 10. Firmante pendiente por sobre

Para los sobres enviados y aún no completados, se identifica cuál es el **próximo firmante pendiente**: el destinatario con el menor `Orden_Envio` cuya acción todavía está en estado `Creado`, `Enviado` o `Entregado`. Esto permite mostrar en el dashboard quién tiene el sobre en su poder sin haberlo firmado.  

Posteriormente, se categoriza ese firmante (cliente, sala de ventas, tramitador, archivo, etc.) para facilitar el análisis de cuellos de botella en el proceso de firmas.

In [143]:
# ============================================================
# FIRMANTE PENDIENTE POR SOBRE
# ============================================================

acciones_pendientes = ['Creado', 'Enviado', 'Entregado']

firmante_pendiente = (
    df[df['Accion'].isin(acciones_pendientes)]
    .sort_values(['ID_Sobre', 'Orden_Envio'])
    .groupby('ID_Sobre')
    .first()
    .reset_index()[['ID_Sobre', 'Destinatario', 'Orden_Envio', 'Email_Destinatario']]
    .rename(columns={
        'Destinatario'      : 'Firmante_Pendiente',
        'Orden_Envio'       : 'Orden_Pendiente',
        'Email_Destinatario': 'Email_Firmante_Pendiente'
    })
)

df = df.merge(firmante_pendiente, on='ID_Sobre', how='left')

print("✅ Firmante pendiente identificado")
print(df[df['Estado']=='Enviado'][['ID_Sobre','Proyecto','Estado','Firmante_Pendiente','Orden_Pendiente']].drop_duplicates('ID_Sobre').head(10).to_string())

✅ Firmante pendiente identificado
                                ID_Sobre         Proyecto   Estado          Firmante_Pendiente  Orden_Pendiente
0   d0911fc7-9c91-4185-9d38-f5750d10ff4d     Entrebosques  Enviado  BUSTAMANTE MUNERA YAMILETH              1.0
4   958db75a-5a02-4123-b145-7fdb10a127f7     Entrebosques  Enviado  BUSTAMANTE MUNERA YAMILETH              1.0
8   ecc815c8-fb17-4a29-b96f-3b33b01f94a2      San Agustin  Enviado    LUZ PATRICIA CAMPO MANCO              1.0
12  989e2ee7-36b8-453c-bb12-4b61d45f2be8    Puerto Sereno  Enviado           Trámites Medellín              2.0
17  3f4b47ff-5d0c-4a73-be1f-6659a55420a2  Puerto Vallarta  Enviado           Trámites Medellín              2.0
21  d9d92bcb-b90e-46a9-aedd-4d3816b0c78f     Puerto Capri  Enviado     Valentina Garcia Ortega              1.0
26  4527944e-e68b-4cce-a38e-0fa10cde9493  Puerto Vallarta  Enviado                  Juan Perez              1.0
31  e4e0e6c6-6d44-4766-b092-505946261d8f      Verde Olivo  Enviado   V

In [144]:
# ============================================================
# CATEGORIZACIÓN DE FIRMANTE PENDIENTE
# ============================================================

def categorizar_firmante(firmante):
    if pd.isna(firmante) or firmante == '':
        return 'Sin firmante'
    f = str(firmante).strip().lower()
    if 'trámites' in f or 'tramites' in f:
        return 'Trámites Medellín'
    if 'sala de ventas' in f or 'sala de negocios' in f:
        return 'Sala de Ventas'
    if '@constructoracapital.com' in f:
        return 'Sala de Ventas'
    if 'tramitador' in f or any(n in f for n in ['karen giraldo', 'claudia pineda', 'daniel ortega', 'esteban marín', 'esteban marin']):
        return 'Tramitador'
    if 'archivo' in f or 'lina marcela betancur' in f:
        return 'Archivo'
    if 'paola estrada' in f:
        return 'Entregas'
    if 'cartera' in f or 'andres lopez' in f or 'recaudo' in f:
        return 'Cartera y Recaudo'
    if 'obra' in f:
        return 'Obra'
    if 'oficiales de cumplimiento' in f:
        return 'Oficiales de Cumplimiento'
    if any(x in f for x in ['enrique echeverri', 'sandra bedoya', 'osman enrique', 'osman páez', 'farid']):
        return 'Dirección Comercial'
    if f in ['san agustín', 'san agustin']:
        return 'Sala de Ventas'
    return 'Cliente'

df['Categoria_Firmante'] = df['Firmante_Pendiente'].apply(categorizar_firmante)

print("✅ Categorías de firmante calculadas")
print(df[df['Estado']=='Enviado']['Categoria_Firmante'].value_counts().to_string())

✅ Categorías de firmante calculadas
Categoria_Firmante
Trámites Medellín            4469
Cliente                      2008
Sala de Ventas                204
Tramitador                    169
Dirección Comercial            89
Sin firmante                   69
Obra                           62
Cartera y Recaudo              40
Oficiales de Cumplimiento      27
Archivo                        18
Entregas                       10


## 11. Semáforo de tiempos

Se implementa una lógica de semáforo (🟢 Verde / 🟡 Amarillo / 🔴 Rojo) que clasifica cada sobre según el tiempo transcurrido respecto a límites establecidos por tipo de documento:
- Para sobres **completados**: se usa `Tiempo_Dias` (días desde envío hasta completado).
- Para sobres **enviados (en proceso)**: se calculan los días transcurridos desde `Fecha_Envio` hasta hoy, dando una señal en tiempo real del estado del trámite.

Esta columna es la principal métrica de gestión operativa del dashboard.

In [145]:
# ============================================================
# SEMÁFORO - Enviados usan días transcurridos
# ============================================================

from datetime import date

# Tiempos límite por código
limites = {
    '120101': (5, 8),
    '120102': (8, 10),
    '120103': (8, 10),
    '120104': (8, 10),
    '120106': (8, 10),
    '120107': (8, 10),
    '120112': (8, 10),
    '120115': (8, 10),
}

def semaforo_v2(row):
    codigo = str(row['Codigo'])[:6]
    estado = row['Estado']
    
    if codigo not in limites:
        return 'Sin límite'
    
    verde_limite, rojo_limite = limites[codigo]
    
    if estado == 'Completado' and not pd.isna(row['Tiempo_Dias']):
        dias = row['Tiempo_Dias']
    elif estado == 'Enviado' and not pd.isna(row['Fecha_Envio']):
        dias = (pd.Timestamp.today() - pd.to_datetime(row['Fecha_Envio'])).days
    else:
        return 'Sin datos'
    
    if dias <= verde_limite:
        return '🟢 Verde'
    elif dias <= rojo_limite:
        return '🟡 Amarillo'
    else:
        return '🔴 Rojo'

df['Semaforo'] = df.apply(semaforo_v2, axis=1)

print("✅ Semáforo v2 calculado")
print(f"\nDistribución sobres Enviados:")
print(df[df['Estado']=='Enviado']['Semaforo'].value_counts().to_string())
print(f"\nDistribución sobres Completados:")
print(df[df['Estado']=='Completado']['Semaforo'].value_counts().to_string())

✅ Semáforo v2 calculado

Distribución sobres Enviados:
Semaforo
🔴 Rojo        6931
Sin límite     234

Distribución sobres Completados:
Semaforo
🔴 Rojo        13299
🟢 Verde        8255
🟡 Amarillo     3032
Sin límite     2974


## 12. Exportación del CSV principal

Se exporta el dataset limpio y enriquecido como `Sobres_Limpio.csv`. Las fechas se convierten al formato ISO `YYYY-MM-DD` para que Power BI las reconozca automáticamente como tipo fecha. Se agrega la columna `Año_Mes` (formato `YYYY-MM`) para facilitar agrupaciones temporales en el dashboard.

**Columnas del CSV final (29 columnas):** `ID_Sobre`, `Proyecto`, `Torre`, `Unidad`, `Tipo_Documento`, `Codigo`, `Asunto`, `Estado`, `Accion`, `Orden_Envio`, `Remitente`, `Destinatario`, `Email_Destinatario`, `Email_Remitente`, `Fecha_Creacion`, `Fecha_Envio`, `Fecha_Completado_Firmante`, `Fecha_Completado_Sobre`, `Tiempo_Dias`, `Num_Devoluciones`, `Tipo_Error`, `Requiere_Seguimiento`, `Requiere_Calidad`, `Grupo`, `Año_Mes`, `Firmante_Pendiente`, `Orden_Pendiente`, `Email_Firmante_Pendiente`, `Categoria_Firmante`, `Semaforo`.

In [146]:
# ============================================================
# EXPORTAR CSV PRINCIPAL CON TODAS LAS COLUMNAS
# ============================================================

df['Año_Mes'] = pd.to_datetime(df['Fecha_Envio'], errors='coerce').dt.strftime('%Y-%m')

df_export2 = df[[
    'ID_Sobre', 'Proyecto', 'Torre', 'Unidad',
    'Tipo_Documento', 'Codigo', 'Asunto', 'Estado', 'Accion',
    'Orden_Envio', 'Remitente', 'Destinatario',
    'Email_Destinatario', 'Email_Remitente',
    'Fecha_Creacion', 'Fecha_Envio',
    'Fecha_Completado_Firmante', 'Fecha_Completado_Sobre',
    'Tiempo_Dias', 'Num_Devoluciones', 'Tipo_Error',
    'Requiere_Seguimiento', 'Requiere_Calidad', 'Grupo', 'Año_Mes',
    'Firmante_Pendiente', 'Orden_Pendiente', 'Email_Firmante_Pendiente',
    'Categoria_Firmante', 'Semaforo'
]].copy()

# Fechas en formato ISO
for col in ['Fecha_Creacion', 'Fecha_Envio',
            'Fecha_Completado_Firmante', 'Fecha_Completado_Sobre']:
    df_export2[col] = pd.to_datetime(df_export2[col], errors='coerce').dt.strftime('%Y-%m-%d')

# Enteros
df_export2['Orden_Pendiente'] = df_export2['Orden_Pendiente'].fillna(0).astype(int)
df_export2['Tipo_Error'] = pd.to_numeric(df_export2['Tipo_Error'], errors='coerce').fillna(0).astype(int)

ruta_export = r'C:\Users\diana\Documents\BI_Sobres\Sobres_Limpio.csv'
df_export2.to_csv(ruta_export, index=False, encoding='utf-8-sig')

print("✅ CSV exportado con Semáforo")
print(f"   Filas:    {df_export2.shape[0]:,}")
print(f"   Columnas: {df_export2.shape[1]}")

✅ CSV exportado con Semáforo
   Filas:    39,277
   Columnas: 30


## 13. Serie de tiempo y predicción de volumen

Se construye una serie temporal que cuenta cuántos sobres únicos se enviaron por mes y por tipo de documento (`Codigo`). Se agrega un promedio móvil de 3 meses para suavizar la tendencia.

Con esta serie se genera una **predicción lineal** para los próximos 6 meses usando regresión de primer grado (`np.polyfit`). El mes actual se excluye del entrenamiento para evitar que los datos incompletos del mes en curso sesguen la proyección.  

El resultado se exporta como `Prediccion_Sobres.csv` con una columna `Tipo` que distingue entre datos históricos y predicción, y un campo `Orden_Mes` numérico para ordenar correctamente el eje X en Power BI.

In [147]:
# ============================================================
# SERIE DE TIEMPO POR MES Y CÓDIGO
# ============================================================

df['Fecha_Envio_dt'] = pd.to_datetime(df['Fecha_Envio'], errors='coerce')

serie = (df.groupby(['Año_Mes', 'Codigo', 'Tipo_Documento'])['ID_Sobre']
         .nunique()
         .reset_index()
         .rename(columns={'ID_Sobre': 'Total_Sobres'}))

serie = serie.sort_values(['Codigo', 'Año_Mes'])

serie['Promedio_Movil'] = (serie.groupby('Codigo')['Total_Sobres']
                           .transform(lambda x: x.rolling(3, min_periods=1).mean().round(0)))

print("✅ Serie de tiempo calculada")
print(f"\nMeses disponibles: {serie['Año_Mes'].nunique()}")
print(serie[serie['Codigo']=='120101'].to_string())

✅ Serie de tiempo calculada

Meses disponibles: 7
    Año_Mes  Codigo   Tipo_Documento  Total_Sobres  Promedio_Movil
1   2025-11  120101  Cierre de venta           377           377.0
13  2025-12  120101  Cierre de venta           344           360.0
27  2026-01  120101  Cierre de venta           415           379.0
38  2026-02  120101  Cierre de venta           392           384.0
51  2026-03  120101  Cierre de venta           560           456.0
63  2026-04  120101  Cierre de venta           352           435.0
76  2026-05  120101  Cierre de venta            30           314.0


In [148]:
# ============================================================
# PREDICCIÓN - Excluir mes actual incompleto
# ============================================================

from datetime import datetime

mes_actual = pd.Timestamp.today().strftime('%Y-%m')

predicciones = []
codigos = serie['Codigo'].unique()

for codigo in codigos:
    df_cod = serie[
        (serie['Codigo'] == codigo) & 
        (serie['Año_Mes'] < mes_actual)
    ].copy().reset_index(drop=True)
    
    if len(df_cod) == 0:
        continue
        
    tipo_doc = df_cod['Tipo_Documento'].iloc[0]
    ultimo_mes = df_cod['Año_Mes'].max()
    
    x = np.arange(len(df_cod))
    y = df_cod['Total_Sobres'].values.astype(float)
    
    if len(x) >= 2:
        coef = np.polyfit(x, y, 1)
        pendiente, intercepto = coef
    else:
        pendiente, intercepto = 0, y[0]
    
    ultimo_dt = pd.to_datetime(ultimo_mes + '-01')
    for i in range(1, 7):
        mes_futuro = (ultimo_dt + pd.DateOffset(months=i)).strftime('%Y-%m')
        x_futuro   = len(df_cod) + i - 1
        prediccion = max(0, round(pendiente * x_futuro + intercepto, 0))
        predicciones.append({
            'Año_Mes'       : mes_futuro,
            'Codigo'        : codigo,
            'Tipo_Documento': tipo_doc,
            'Total_Sobres'  : None,
            'Prediccion'    : prediccion,
            'Tipo'          : 'Predicción'
        })

serie_historico = serie[serie['Año_Mes'] < mes_actual].copy()
serie_historico['Prediccion'] = serie_historico['Total_Sobres']
serie_historico['Tipo'] = 'Histórico'

df_pred = pd.concat([serie_historico, pd.DataFrame(predicciones)], ignore_index=True)
df_pred = df_pred.sort_values(['Codigo', 'Año_Mes'])

print("✅ Predicción calculada")
print(df_pred[df_pred['Codigo']=='120101'][['Año_Mes','Total_Sobres','Prediccion','Tipo']].to_string())

✅ Predicción calculada
    Año_Mes Total_Sobres  Prediccion        Tipo
2   2025-11          377       377.0   Histórico
3   2025-12          344       344.0   Histórico
4   2026-01          415       415.0   Histórico
5   2026-02          392       392.0   Histórico
6   2026-03          560       560.0   Histórico
7   2026-04          352       352.0   Histórico
88  2026-05         None       457.0  Predicción
89  2026-06         None       471.0  Predicción
90  2026-07         None       485.0  Predicción
91  2026-08         None       500.0  Predicción
92  2026-09         None       514.0  Predicción
93  2026-10         None       528.0  Predicción


In [149]:
# ============================================================
# EXPORTAR CSV DE PREDICCIÓN CON ORDEN_MES
# ============================================================

df_pred_export = df_pred[[
    'Año_Mes', 'Codigo', 'Tipo_Documento', 
    'Total_Sobres', 'Prediccion', 'Tipo'
]].copy()

meses_unicos = sorted(df_pred_export['Año_Mes'].unique())
orden_mes = {mes: i+1 for i, mes in enumerate(meses_unicos)}
df_pred_export['Orden_Mes'] = df_pred_export['Año_Mes'].map(orden_mes)

ruta_pred = r'C:\Users\diana\Documents\BI_Sobres\Prediccion_Sobres.csv'
df_pred_export.to_csv(ruta_pred, index=False, encoding='utf-8-sig')

print("✅ CSV de predicción exportado con Orden_Mes")
print(f"   Filas:    {df_pred_export.shape[0]:,}")
print(f"   Columnas: {df_pred_export.shape[1]}")
print(f"\nDistribución:")
print(df_pred_export['Tipo'].value_counts())

✅ CSV de predicción exportado con Orden_Mes
   Filas:    184
   Columnas: 7

Distribución:
Tipo
Predicción    108
Histórico      76
Name: count, dtype: int64


## 14. Proyección con Suavizamiento Exponencial (Holt's Method)

Para la proyección de volumen de sobres evaluamos distintos enfoques:

- **Modelos de ML** (Random Forest, redes neuronales): descartados porque requieren mínimo 2–3 años de datos para generalizar bien. Con pocos meses solo aprenden ruido.
- **Regresión lineal simple**: descartada porque asume que la tendencia es perfectamente recta, lo cual es muy sensible al punto de inicio y de corte.
- **Suavizamiento Exponencial de Holt (doble)**: seleccionado. Es el estándar estadístico para series de tiempo cortas con tendencia. Le da más peso a los datos recientes, y permite calcular un intervalo de confianza que comunica honestamente la incertidumbre de la proyección.

El resultado se exporta en `Prediccion_Sobres.csv` con columnas de límite inferior y superior para visualizar la banda de confianza en Power BI.

In [150]:
# ============================================================
# 14. PROYECCIÓN - Suavizamiento Exponencial de Holt
# ============================================================

# --- Parámetros ---
MESES_PROYECCION = 6      # cuántos meses hacia adelante proyectar
INTERVALO_CONF   = 0.90   # nivel de confianza de la banda (90%)

# --- Construir serie mensual por tipo de documento ---
# Se excluye el mes actual porque puede estar incompleto
mes_actual = pd.Timestamp.today().strftime('%Y-%m')

serie_base = (
    df[df['Año_Mes'] < mes_actual]
    .drop_duplicates('ID_Sobre')[['ID_Sobre', 'Año_Mes', 'Codigo', 'Tipo_Documento']]
    .groupby(['Año_Mes', 'Codigo', 'Tipo_Documento'])
    .size()
    .reset_index(name='Total_Sobres')
    .sort_values(['Codigo', 'Año_Mes'])
)

print(f"Meses disponibles en histórico: {serie_base['Año_Mes'].nunique()}")
print(f"Tipos de documento: {serie_base['Codigo'].nunique()}")
print(serie_base.groupby('Codigo')['Total_Sobres'].count().rename('meses_con_datos'))

Meses disponibles en histórico: 6
Tipos de documento: 18
Codigo
120069    1
12010.    1
120101    6
120102    6
120103    6
120104    6
120105    1
120106    6
120107    6
120108    5
120109    5
120110    2
120112    6
120115    5
120124    6
120125    6
Comple    1
FORMUL    1
Name: meses_con_datos, dtype: int64


In [151]:
# ============================================================
# Ajuste del modelo y generación de proyección por código
# ============================================================

resultados = []

for codigo in serie_base['Codigo'].unique():

    df_cod = serie_base[serie_base['Codigo'] == codigo].copy().reset_index(drop=True)
    tipo_doc = df_cod['Tipo_Documento'].iloc[0]
    y = df_cod['Total_Sobres'].values.astype(float)
    meses = df_cod['Año_Mes'].tolist()

    # --- Guardar histórico ---
    for i, mes in enumerate(meses):
        resultados.append({
            'Año_Mes'       : mes,
            'Codigo'        : codigo,
            'Tipo_Documento': tipo_doc,
            'Total_Sobres'  : int(y[i]),
            'Proyeccion'    : int(y[i]),
            'Limite_Inf'    : None,
            'Limite_Sup'    : None,
            'Tipo'          : 'Histórico'
        })

    # --- Ajustar modelo ---
    # Si hay menos de 3 meses usamos promedio simple; si hay más usamos Holt
    if len(y) < 3:
        # Muy pocos datos: proyectamos con el promedio de lo disponible
        prom = y.mean()
        std  = y.std() if len(y) > 1 else prom * 0.3
        z    = stats.norm.ppf((1 + INTERVALO_CONF) / 2)
        for i in range(1, MESES_PROYECCION + 1):
            mes_fut = (pd.to_datetime(meses[-1] + '-01') + pd.DateOffset(months=i)).strftime('%Y-%m')
            resultados.append({
                'Año_Mes'       : mes_fut,
                'Codigo'        : codigo,
                'Tipo_Documento': tipo_doc,
                'Total_Sobres'  : None,
                'Proyeccion'    : max(0, round(prom, 1)),
                'Limite_Inf'    : max(0, round(prom - z * std, 1)),
                'Limite_Sup'    : round(prom + z * std, 1),
                'Tipo'          : 'Proyección'
            })
    else:
        # Holt's Double Exponential Smoothing (nivel + tendencia)
        modelo = ExponentialSmoothing(
            y,
            trend='add',          # captura la tendencia
            damped_trend=True,    # amortigua la tendencia para no exagerar
            initialization_method='estimated'
        ).fit(optimized=True)    # deja que el modelo elija los mejores parámetros

        forecast = modelo.forecast(MESES_PROYECCION)

        # Intervalo de confianza basado en el error del modelo en histórico
        residuos = modelo.resid
        std_err  = residuos.std()
        z        = stats.norm.ppf((1 + INTERVALO_CONF) / 2)

        for i in range(MESES_PROYECCION):
            mes_fut  = (pd.to_datetime(meses[-1] + '-01') + pd.DateOffset(months=i+1)).strftime('%Y-%m')
            # El intervalo se amplía con el tiempo (más incertidumbre a futuro)
            margen   = z * std_err * np.sqrt(i + 1)
            proy     = max(0, round(forecast[i], 1))
            resultados.append({
                'Año_Mes'       : mes_fut,
                'Codigo'        : codigo,
                'Tipo_Documento': tipo_doc,
                'Total_Sobres'  : None,
                'Proyeccion'    : proy,
                'Limite_Inf'    : max(0, round(proy - margen, 1)),
                'Limite_Sup'    : round(proy + margen, 1),
                'Tipo'          : 'Proyección'
            })

df_proyeccion = pd.DataFrame(resultados).sort_values(['Codigo', 'Año_Mes']).reset_index(drop=True)

# Columna de orden para Power BI
meses_ord = {m: i+1 for i, m in enumerate(sorted(df_proyeccion['Año_Mes'].unique()))}
df_proyeccion['Orden_Mes'] = df_proyeccion['Año_Mes'].map(meses_ord)

print("✅ Proyección calculada")
print(f"\nRegistros históricos: {(df_proyeccion['Tipo']=='Histórico').sum()}")
print(f"Registros proyectados: {(df_proyeccion['Tipo']=='Proyección').sum()}")
print()
print("--- Muestra proyección código 120101 ---")
print(df_proyeccion[df_proyeccion['Codigo']=='120101'][
    ['Año_Mes','Total_Sobres','Proyeccion','Limite_Inf','Limite_Sup','Tipo']
].to_string(index=False))

✅ Proyección calculada

Registros históricos: 76
Registros proyectados: 108

--- Muestra proyección código 120101 ---
Año_Mes  Total_Sobres  Proyeccion  Limite_Inf  Limite_Sup       Tipo
2025-11         377.0       377.0         NaN         NaN  Histórico
2025-12         344.0       344.0         NaN         NaN  Histórico
2026-01         415.0       415.0         NaN         NaN  Histórico
2026-02         392.0       392.0         NaN         NaN  Histórico
2026-03         560.0       560.0         NaN         NaN  Histórico
2026-04         352.0       352.0         NaN         NaN  Histórico
2026-05           NaN       447.7       336.8       558.6 Proyección
2026-06           NaN       453.9       297.1       610.7 Proyección
2026-07           NaN       458.7       266.6       650.8 Proyección
2026-08           NaN       462.6       240.8       684.4 Proyección
2026-09           NaN       465.8       217.8       713.8 Proyección
2026-10           NaN       468.3       196.7       73

In [152]:
# ============================================================
# Exportar CSV de proyección
# ============================================================

ruta_pred = r'C:\Users\diana\OneDrive\Escritorio\Proyecto_1\BI_Sobres\Prediccion_Sobres.csv'

df_proyeccion.to_csv(ruta_pred, index=False, encoding='utf-8-sig')

print("✅ Prediccion_Sobres.csv exportado")
print(f"   Filas:    {df_proyeccion.shape[0]}")
print(f"   Columnas: {list(df_proyeccion.columns)}")

✅ Prediccion_Sobres.csv exportado
   Filas:    184
   Columnas: ['Año_Mes', 'Codigo', 'Tipo_Documento', 'Total_Sobres', 'Proyeccion', 'Limite_Inf', 'Limite_Sup', 'Tipo', 'Orden_Mes']


In [153]:
print(df_pred_export.columns.tolist())
print(df_pred_export.head(3))

['Año_Mes', 'Codigo', 'Tipo_Documento', 'Total_Sobres', 'Prediccion', 'Tipo', 'Orden_Mes']
    Año_Mes  Codigo                 Tipo_Documento Total_Sobres  Prediccion  \
0   2026-04  120069  Tipo pendiente por clasificar            1         1.0   
76  2026-05  120069  Tipo pendiente por clasificar         None         1.0   
77  2026-06  120069  Tipo pendiente por clasificar         None         1.0   

          Tipo  Orden_Mes  
0    Histórico          6  
76  Predicción          7  
77  Predicción          8  


---

## ✅ Resumen del pipeline

| Paso | Descripción | Columnas generadas |
|------|-------------|--------------------|
| 1 | Carga del CSV con detección de separador | DataFrame base |
| 2 | Renombrado de columnas | Nombres en español |
| 3–5 | Conversión y corrección de fechas (`dayfirst=True`) | `Fecha_Completado_Sobre`, `Tiempo_Dias` |
| 6 | Clasificación por tipo de documento | `Tipo_Documento`, `Codigo`, `Marcacion_Error`, `Num_Devoluciones`, `Tipo_Error` |
| 7 | Extracción de proyecto, torre y unidad | `Proyecto`, `Torre`, `Unidad` |
| 8 | Mapeo a nombres oficiales de proyectos | `Proyecto` (normalizado) |
| 9 | Flags de gestión | `Requiere_Seguimiento`, `Requiere_Calidad` |
| 10 | Identificación del firmante pendiente | `Firmante_Pendiente`, `Categoria_Firmante` |
| 11 | Semáforo de tiempos por tipo de documento | `Semaforo` |
| 12 | Exportación CSV principal | `Sobres_Limpio.csv` |
| 13 | Serie de tiempo + predicción lineal | `Prediccion_Sobres.csv` |